# Bab 2: Data Inspection (Inspeksi Data) - Pondasi AI/ML Engineer

## A. Definisi & Tujuan Utama
**Data Inspection** adalah langkah pertama setelah me-load dataset untuk memahami bentuk fisik dan kesehatan statistik data tanpa harus membuka keseluruhan file.

> 💡 **Aturan Emas ML:** *"Garbage In, Garbage Out"* (Jika data yang masuk kotor/sampah, maka model ML yang dihasilkan juga akan buruk/bodoh). Sekitar 70%-80% waktu seorang AI/ML Engineer dihabiskan di tahap *Data Preprocessing* ini.

---

## B. Syntax Dasar Inspeksi Fisik Data

* `df.head(n)`
    * **Fungsi:** Menampilkan $n$ baris pertama (secara *default* 5 baris).
    * **Tujuan ML:** Mengintip contoh fisik isi data dan nama-nama kolom.
* `df.tail(n)`
    * **Fungsi:** Menampilkan $n$ baris terakhir.
    * **Tujuan ML:** Memastikan konsistensi baris akhir data (apakah ada baris rusak di paling bawah).
* `df.shape`
    * **Fungsi:** Mengembalikan *tuple* berupa `(jumlah_baris, jumlah_kolom)`.
    * **Tujuan ML:** Mengetahui dimensi ukuran dataset sebelum masuk ke proses *training*.
* `df.columns`
    * **Fungsi:** Menampilkan daftar nama semua kolom yang tersedia dalam bentuk list.

---

## C. Syntax Diagnostik Kesehatan Data

### 1. `df.info()`
Memberikan ringkasan metadata teknis: `RangeIndex` (total baris), `Non-Null Count` (data yang terisi/tidak kosong), dan `Dtype` (tipe data per kolom seperti *int64, float64, object/teks*).
* **Poin ML:** Jika `Non-Null Count` lebih kecil dari total baris, artinya ada kolom yang bocor atau memiliki data kosong (`NaN`).

### 2. `df.isnull().sum()`
Menghitung jumlah data kosong (`NaN`) secara instan untuk setiap kolom.
* **Poin ML:** Algoritma Machine Learning benci data kosong (`NaN`). Ini adalah alarm utama untuk masuk ke tahapan *Data Cleaning*.

### 3. `df.describe()`
Menghitung statistik deskriptif otomatis untuk kolom numerik (`count`, `mean`, `std`, `min`, `max`, dan persentil `25%`, `50%`, `75%`).
* **Deteksi Outlier:** Periksa nilai `min` dan `max` untuk mencari keanehan (misal: usia pasien bernilai $-5$ atau $999$).
* **Feature Scaling:** Jika rentang antar kolom berbeda jauh (misal Usia berkisar $20-50$, sedangkan Pendapatan berkisar di angka jutaan), data **WAJIB** disamakan skalanya (*Normalization/Standardization*) pada tahap *Data Transformation* nanti agar model ML tidak bias karena perhitungan jarak matematika.

---

## D. Inspeksi Lanjutan Khusus AI/ML

### 1. Cek Imbalance Class
```python
df['nama_kolom'].value_counts()
# atau untuk persentase:
df['nama_kolom'].value_counts(normalize=True) * 100
```
* **Analisis:** Digunakan pada kolom *Target* ($y$) untuk kasus klasifikasi. Jika data tidak seimbang *(misal: 990 pasien Sehat vs 10 Sakit Jantung)*, model ML akan menjadi "malas" dan cenderung menebak kelas mayoritas saja karena merasa akurasinya sudah tinggi ($99\%$).
* **Solusi:** 
    1. **Resampling** (Diduplikasi/menggunakan algoritma tiruan SMOTE).
    2. **Class Weight** (Memberikan bobot hukuman nilai lebih berat jika model salah menebak kelas minoritas).
    3.  Mengganti indikator ujian dari Accuracy menjadi Recall atau F1-Score.

### 2. Matrix Korelasi (`df.corr()`)
Menghitung hubungan linear antara dua variabel angka dengan rentang nilai dari $-1$ hingga $+1$.
* **Garis Diagonal:** Bernilai `1.00` karena membandingkan kolom yang sama dengan dirinya sendiri (Bisa diabaikan).
* **Cara Baca Seleksi Fitur:**
    * **Fitur ($X$) vs Target ($y$):** Cari nilai mutlak yang mendekati 1 $\rightarrow$ Artinya Fitur Kuat dan wajib dipertahankan.
    * **Fitur ($X$) vs Fitur ($X$):** Jika korelasi antar fitur $> 0.80$ $\rightarrow$ Terjadi Multikolinieritas (Fitur Kembar/Tumpang Tindih). Solusi: Buang salah satu fitur yang nilai korelasinya terhadap target utama lebih lemah.
* **⚠️ PENTING:** Posisi Target ($y$) hanya ditentukan oleh **GOAL PROYEK** sejak awal (apa yang ingin kamu tebak). Angka korelasi yang kuat tidak akan pernah bisa mengubah status sebuah fitur menjadi target.

---

In [1]:
import pandas as pd
import numpy as np

In [2]:
# head() -> menampilkan n baris pertama dari DataFrame
# tail() -> menampilkan n baris terakhir dari DataFrame

# contoh dataset
data = {
    "Nama": ["Ali", "Budi", "Cici", "Dedi", "Evi", "Fahri", "Gita", "Hadi", "Indah", "Joko"],
    "Usia": [23, 45, np.nan, 35, 29, 52, 40, np.nan, 31, 48],
    "Tekanan_Darah": [120, 140, 115, 130, 110, 145, 125, 135, 118, 150],
    "Penyakit_Jantung": [0, 1, 0, 0, 0, 1, 0, 1, 0, 1]
}

df_pasien = pd.DataFrame(data)

# menampilkan 3 baris pertama
display(df_pasien.head(3))

# menampilkan 3 baris terakhir
display(df_pasien.tail(3))

,Nama,Usia,Tekanan_Darah,Penyakit_Jantung
0,Ali,23.0,120,0
1,Budi,45.0,140,1
2,Cici,NaN,115,0


,Nama,Usia,Tekanan_Darah,Penyakit_Jantung
7,Hadi,NaN,135,1
8,Indah,31.0,118,0
9,Joko,48.0,150,1


In [3]:
# memeriksa atau merangkum teknis mengenai DataFrame
df_pasien.info()

<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 4 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Nama              10 non-null     str    
 1   Usia              8 non-null      float64
 2   Tekanan_Darah     10 non-null     int64  
 3   Penyakit_Jantung  10 non-null     int64  
dtypes: float64(1), int64(2), str(1)
memory usage: 452.0 bytes


In [4]:
# memberikan gambaran berupa statsitik bertipe numerik 
# berguna untuk mendeteksi outliner/anomali
df_pasien.describe()

,Usia,Tekanan_Darah,Penyakit_Jantung
count,8.000000,10.000000,10.000000
mean,37.875000,128.800000,0.400000
std,10.091545,13.456101,0.516398
min,23.000000,110.000000,0.000000
25%,30.500000,118.500000,0.000000
50%,37.500000,127.500000,0.000000
75%,45.750000,138.750000,1.000000
max,52.000000,150.000000,1.000000


In [9]:
# memeriksa dimensi kolom (jumlah baris, jumlah kolom)
df_pasien.shape

(10, 4)

In [11]:
# memeriksa isi dari kolom -> .columns
df_pasien.columns.tolist()

['Nama', 'Usia', 'Tekanan_Darah', 'Penyakit_Jantung']

In [ ]:
# mendeteksi sejumlah missing value -> .isnull()
df_pasien.isnull().sum()

Nama                0
Usia                2
Tekanan_Darah       0
Penyakit_Jantung    0
dtype: int64

In [ ]:
# memeriksa keseimbangan data
display(df_pasien["Penyakit_Jantung"].value_counts())
display(df_pasien["Penyakit_Jantung"].value_counts(normalize=True)* 100)

Penyakit_Jantung
0    6
1    4
Name: count, dtype: int64

Penyakit_Jantung
0    60.0
1    40.0
Name: proportion, dtype: float64

In [20]:
kolom_numerik = df_pasien.select_dtypes(include=["int64", "float64"])
matriks_korelasi = kolom_numerik.corr()

display(matriks_korelasi)

,Usia,Tekanan_Darah,Penyakit_Jantung
Usia,1.000000,0.886685,0.858176
Tekanan_Darah,0.886685,1.000000,0.876263
Penyakit_Jantung,0.858176,0.876263,1.000000
